In [ ]:
import pandas
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import *
from torch import nn
import time
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')
def print_loss(epoch, num_epochs, batch_index, num_batchs, args):
    n_progs = 30
    print(f"Epoch:{epoch}/{num_epochs}, Batch:{batch_index}/{num_batchs}", end = ' ')
    print('<', end = '')
    index = (batch_index // (num_batchs // n_progs))
    for i in range(n_progs):
        if i == index:
            print('>', end = '')
        elif i > index:
            print('.', end = '')
        else:
            print('=', end = '')
    print('>', end = ' , ')
    for i, (title, value) in enumerate(args.items()):
        print(f"{title} = {value:2f}", end = ' , ' if i < len(args) - 1 else '')
    clear_output(wait = True)
    time.sleep(0.2)

In [ ]:
dataset = pandas.read_csv(r"/kaggle/input/digit-recognizer/train.csv")
dataset

In [ ]:
images = dataset.iloc[:, 1:].values.reshape((-1, 28, 28)) / 255.0
labels = dataset.iloc[:, 0].values
images.shape, labels.shape

In [ ]:
stamp_images = [images[labels == i][:10] for i in range(len(np.unique(labels)))]
stamp_images[0].shape

In [ ]:
class PairwiseDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, transform):
        super().__init__()
        self.data = X
        self.target = y
        self.clas_images = [self.data[self.target == i] for i in range(len(np.unique(self.target)))]
        self.transform = transform
    
    def __len__(self):
        return len(images)
    
    def __getitem__(self, index):
        x1 = self.data[index]
        lbl = self.target[index]
        x2 = None
        y = None
        prob = np.random.randint(1, 101)
        if prob <= 50:
            x2 = self.clas_images[lbl][np.random.randint(1, 3000)]
            y = torch.tensor([1.0])
        else:
            not_same = np.random.randint(1, len(np.unique(self.target)))
            x2 = self.clas_images[not_same][np.random.randint(1, 3000)]
            if not_same == lbl:
                y = torch.tensor([1.0])
            else:
                y = torch.tensor([0.0])
        x1 = self.transform(x1.astype(np.float32))
        x2 = self.transform(x2.astype(np.float32))
        return x1, x2, y

In [ ]:
pairwise_dataset = PairwiseDataset(images, labels, transforms.ToTensor())

In [ ]:
train_loader = torch.utils.data.DataLoader(pairwise_dataset, shuffle = True, batch_size = 100)
a, b, c = next(iter(train_loader))
a.shape, b.shape, c.shape, c.sum()

In [ ]:
def show_image(image1:torch.tensor, image2:torch.tensor, title = None):
    if not torch.is_tensor(image1):
        image1 = transforms.ToTensor()(image1)
    
    if not torch.is_tensor(image2):
        image2 = transforms.ToTensor()(image2)
    
    grid = torchvision.utils.make_grid([image1, image2])
    grid = grid.numpy().transpose((1, 2, 0))
    plt.imshow(grid)
    plt.title(title)
    plt.xticks([])
    plt.yticks([])
for i in range(16):
    plt.subplot(4, 4, i + 1)
    show_image(a[i], b[i], c[i].item())
plt.show()

In [ ]:
class DistanceLayer(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, a, b):
        return torch.abs(a - b)

class SiameseNetwork(nn.Module):
    def __init__(self, device = 'cuda'):
        super().__init__()
        self.optimizer = None
        self.criterion = None
        self.device = device
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size = (3, 3), stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            
            nn.Conv2d(64, 64, kernel_size = (3, 3), stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size = (2, 2), stride = 2),
            
            nn.Conv2d(64, 128, kernel_size = (3, 3), stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            
            nn.Conv2d(128, 128, kernel_size = (3, 3), stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size = (2, 2), stride = 2),
        
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 32),
        )
        
        self.output_layer = nn.Sequential(
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
        self.distance_layer = DistanceLayer()
        
    def encode(self, x):
        return self.encoder(x)
    
    def forward(self, a, b):
        encoded1, encoded2 = self.encode(a), self.encode(b)
        distance = self.distance_layer(encoded1, encoded2)
        return self.output_layer(distance)
    
    def compiler(self, optimizer, loss):
        self.optimizer = optimizer
        self.criterion = loss
        
    def predict(self, x1, x2):
        return np.array([self.predict_sample(it1, it2) for it1,it2 in list(zip(x1, x2))])
    
    def predict_sample(self, a, b):
        if not torch.is_tensor(a):
            a = transforms.ToTensor()(a.astype(np.float32))
        
        if not torch.is_tensor(b):
            b = transforms.ToTensor()(b.astype(np.float32))
        
        with torch.no_grad():
            y_pred = self.to('cpu').forward(a.reshape(1, *a.shape), b.reshape(1, *b.shape))
            return y_pred.item()
        
    def train_step(self, batch):
        
        x1, x2, y = batch
        x1 = x1.to(self.device)
        x2 = x2.to(self.device)
        y = y.to(self.device)
        
        self.optimizer.zero_grad() 
        y_pred = self.forward(x1, x2)
        loss = self.criterion(y_pred, y)
        
        loss.backward()
        
        self.optimizer.step()
        acc = (y_pred.round() == y).float().mean()
        return {'loss' : loss.item(), 'accuracy' : acc.item()}
    
    def fit(self, train_loader, epochs = 1):
        for epoch in range(epochs):
            for i, batch in enumerate(train_loader):
                train_log = self.train_step(batch)
                print_loss(epoch, epochs, i, len(train_loader), train_log)
                

In [ ]:
model = SiameseNetwork('cuda').to('cuda')
model.compiler(optimizer = torch.optim.Adam(model.parameters(), lr = 0.001), loss = nn.BCELoss())

In [ ]:
model.fit(train_loader, epochs = 5)

In [ ]:
y_pred = model.predict(a, b)
y_pred.shape

In [ ]:
def show_image(image1:torch.tensor, image2:torch.tensor, title = None):
    grid = torchvision.utils.make_grid([image1, image2])
    grid = grid.numpy().transpose((1, 2, 0))
    plt.imshow(grid)
    plt.title(title, color = 'red' if title != 'Good' else 'black')
    plt.xticks([])
    plt.yticks([])
for i in range(36):
    plt.subplot(6, 6, i + 1)
    show_image(a[i], b[i], "Good" if (c[i].item() == y_pred[i].round().item()) else "Bad" )
plt.show()

In [ ]:
def get_most_similar(test_image, stamp_images):
    mx = 0.0
    best = -1
    for i, clas in enumerate(stamp_images):
        probs = []
        for image in clas:
            probs.append(model.predict_sample(test_image, image))
        if np.mean(probs) > mx:
            mx = np.mean(probs)
            best = i
    return best, mx
p = np.random.randint(0, len(c))
get_most_similar(a[34], stamp_images)